# ***## Setup and installation***

In [48]:
!pip install anthropic --quiet

In [49]:
!pip install transformers torch --quiet

In [50]:
import anthropic
from google.colab import userdata

# Load the anthropic API key
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Quick test call using Haiku model
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": "Reply with exactly: connection works"}]
)

print(response.content[0].text)

connection works


# ***## The Triage Prompt (SCAFF Structured)***

In [51]:
# The improved SCAFF-structured triage prompt
TRIAGE_PROMPT = """You are an ESG operations triage assistant for Kaisync. An employee has submitted an operational message. Classify it for routing to the correct team.

Return a single JSON object only. No text before or after it.

Use ONLY these fixed values:

issue_category: one of ["Energy", "Water", "Waste & Recycling", "Facilities & Safety", "Sustainable Procurement", "Accessibility & Inclusion", "Governance & Conduct", "Other"]

recommended_team: one of ["Facilities", "Sustainability", "Procurement", "People & Culture", "Governance", "Health & Safety"]

urgency: one of ["LOW", "MEDIUM", "HIGH", "CRITICAL"], judged as:
  CRITICAL = immediate risk to people, safety, or legal compliance
  HIGH = significant operational or environmental impact, act within 24h
  MEDIUM = should be handled soon but no immediate harm
  LOW = minor or informational

sentiment: one of ["POSITIVE", "NEUTRAL", "NEGATIVE"]

followup_required: "Y" or "N"

data_sensitivity_risk: one of ["NONE", "LOW", "MEDIUM", "HIGH"], where HIGH = personal, wellbeing, whistleblowing, or identifiable individual data

confidence: a number from 0.0 to 1.0 for how certain you are of the classification

needs_human_review: "Y" or "N". Set to "Y" if confidence is below 0.6, if the message contains more than one distinct issue, or if the message is too vague to classify reliably. When unsure, prefer "Y".

escalation_reason: a short phrase explaining any escalation, or "None"

brief_summary: one sentence, neutral, no invented detail

Rules:
- Do not invent information that is not in the message.
- If a field cannot be determined, use your best estimate and lower the confidence.
- Base urgency on real-world consequence, not on emotional tone.

Message: """

# The three test messages
messages = {
    "M1_clear": "There is a water leak in Building C that has been running all morning.",
    "M2_emotional": "The recycling bins are contaminated AGAIN and nobody ever checks them. This is ridiculous.",
    "M3_double": "The accessible entrance has been blocked for two days and honestly I think one of our suppliers is ignoring the sustainability policy too."
}

print("Prompt and messages loaded. Ready to run.")

Prompt and messages loaded. Ready to run.


# ***## Experiment 1: Single-run classification***

In [52]:
import json

def triage(message_text):
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": TRIAGE_PROMPT + message_text}]
    )
    return response.content[0].text

# Run each message once and print the JSON result
for key, msg in messages.items():
    print("=" * 60)
    print(f"{key}: {msg}")
    print("-" * 60)
    result = triage(msg)
    print(result)
    print()

M1_clear: There is a water leak in Building C that has been running all morning.
------------------------------------------------------------
```json
{
  "issue_category": "Water",
  "recommended_team": "Facilities",
  "urgency": "HIGH",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "data_sensitivity_risk": "NONE",
  "confidence": 0.95,
  "needs_human_review": "N",
  "escalation_reason": "Active water waste with extended duration requires prompt facilities response",
  "brief_summary": "Water leak in Building C has been ongoing since morning."
}
```

M2_emotional: The recycling bins are contaminated AGAIN and nobody ever checks them. This is ridiculous.
------------------------------------------------------------
```json
{
  "issue_category": "Waste & Recycling",
  "recommended_team": "Facilities",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "data_sensitivity_risk": "NONE",
  "confidence": 0.85,
  "needs_human_review": "N",
  "escalation_re

# ***## Experiment 2: Consistency across repeated runs***

In [53]:
from collections import Counter
import json, re

def triage_clean(message_text):
    raw = triage(message_text)
    # strip markdown code fences if present
    cleaned = re.sub(r"```json|```", "", raw).strip()
    try:
        return json.loads(cleaned)
    except:
        return None

# Run each message 5 times, track urgency + category + review flag
print("CONSISTENCY TEST: each message run 5 times\n")
for key, msg in messages.items():
    urgencies, categories, reviews = [], [], []
    for i in range(5):
        out = triage_clean(msg)
        if out:
            urgencies.append(out.get("urgency"))
            categories.append(out.get("issue_category"))
            reviews.append(out.get("needs_human_review"))
    print("=" * 55)
    print(f"{key}: {msg[:50]}...")
    print(f"  urgency over 5 runs:     {urgencies}")
    print(f"  category over 5 runs:    {categories}")
    print(f"  needs_review over 5 runs:{reviews}")
    print()

CONSISTENCY TEST: each message run 5 times

M1_clear: There is a water leak in Building C that has been ...
  urgency over 5 runs:     ['HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH']
  category over 5 runs:    ['Water', 'Water', 'Water', 'Water', 'Water']
  needs_review over 5 runs:['N', 'N', 'N', 'N', 'N']

M2_emotional: The recycling bins are contaminated AGAIN and nobo...
  urgency over 5 runs:     ['MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM', 'MEDIUM']
  category over 5 runs:    ['Waste & Recycling', 'Waste & Recycling', 'Waste & Recycling', 'Waste & Recycling', 'Waste & Recycling']
  needs_review over 5 runs:['N', 'N', 'N', 'N', 'N']

M3_double: The accessible entrance has been blocked for two d...
  urgency over 5 runs:     ['HIGH', 'HIGH', 'HIGH', 'HIGH', 'HIGH']
  category over 5 runs:    ['Facilities & Safety', 'Facilities & Safety', 'Accessibility & Inclusion', 'Accessibility & Inclusion', 'Accessibility & Inclusion']
  needs_review over 5 runs:['Y', 'Y', 'Y', 'Y', 'Y']



# ***## Experiment 3: Hugging Face baseline comparison***

In [54]:
from transformers import pipeline
print("transformers imported successfully")

transformers imported successfully


In [55]:
from transformers import pipeline

# Load the zero-shot classifier (downloads the model on first run)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# The same category options the LLM used
candidate_labels = ["Energy", "Water", "Waste & Recycling", "Facilities & Safety",
                    "Sustainable Procurement", "Accessibility & Inclusion",
                    "Governance & Conduct", "Other"]

# Also test urgency as a separate classification
urgency_labels = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

print("BASELINE: Hugging Face zero-shot classification\n")
for key, msg in messages.items():
    cat_result = classifier(msg, candidate_labels)
    urg_result = classifier(msg, urgency_labels)
    print("=" * 55)
    print(f"{key}: {msg[:55]}...")
    print(f"  Top category: {cat_result['labels'][0]}  (score {cat_result['scores'][0]:.2f})")
    print(f"  Top urgency:  {urg_result['labels'][0]}  (score {urg_result['scores'][0]:.2f})")
    print()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

BASELINE: Hugging Face zero-shot classification

M1_clear: There is a water leak in Building C that has been runni...
  Top category: Water  (score 0.81)
  Top urgency:  MEDIUM  (score 0.29)

M2_emotional: The recycling bins are contaminated AGAIN and nobody ev...
  Top category: Waste & Recycling  (score 0.60)
  Top urgency:  HIGH  (score 0.39)

M3_double: The accessible entrance has been blocked for two days a...
  Top category: Sustainable Procurement  (score 0.49)
  Top urgency:  HIGH  (score 0.36)



# ***## The triage system (validation, fallback, splitting)***

In [56]:
from pydantic import BaseModel, field_validator
from typing import Literal

class TriageTicket(BaseModel):
    issue_category: Literal["Energy", "Water", "Waste & Recycling", "Facilities & Safety",
                            "Sustainable Procurement", "Accessibility & Inclusion",
                            "Governance & Conduct", "Other"]
    recommended_team: Literal["Facilities", "Sustainability", "Procurement",
                              "People & Culture", "Governance", "Health & Safety"]
    urgency: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    sentiment: Literal["POSITIVE", "NEUTRAL", "NEGATIVE"]
    followup_required: Literal["Y", "N"]
    data_sensitivity_risk: Literal["NONE", "LOW", "MEDIUM", "HIGH"]
    confidence: float
    needs_human_review: Literal["Y", "N"]
    escalation_reason: str
    brief_summary: str

print("Validation schema defined.")

Validation schema defined.


In [57]:
import json, re

def make_error_ticket(message_text, reason):
    """If anything fails, build a safe ticket routed to manual review. No data is lost."""
    return {
        "issue_category": "Other",
        "recommended_team": "Health & Safety",
        "urgency": "HIGH",
        "sentiment": "NEUTRAL",
        "followup_required": "Y",
        "data_sensitivity_risk": "NONE",
        "confidence": 0.0,
        "needs_human_review": "Y",
        "escalation_reason": f"System could not classify this message ({reason}). Sent to manual queue.",
        "brief_summary": message_text,
        "routing_action": "ESCALATED TO MANUAL QUEUE (system error)"
    }

def triage_validated(message_text):
    """Classify, validate against the schema, and never drop a ticket on failure."""
    try:
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=400,
            messages=[{"role": "user", "content": TRIAGE_PROMPT + message_text}]
        )
        raw = response.content[0].text
        cleaned = re.sub(r"```json|```", "", raw).strip()
        parsed = json.loads(cleaned)
        ticket = TriageTicket(**parsed).model_dump()   # validates here; raises if invalid
    except Exception as e:
        return make_error_ticket(message_text, type(e).__name__)
    # decide routing action
    if ticket["needs_human_review"] == "Y":
        ticket["routing_action"] = "ESCALATED TO HUMAN REVIEW"
    else:
        ticket["routing_action"] = f"AUTO-ROUTED to {ticket['recommended_team']} team"
    return ticket

print("Validated triage function ready.")

Validated triage function ready.


In [58]:
def split_into_issues(message_text):
    """Ask Claude whether the message contains multiple distinct issues; if so, split them."""
    split_prompt = (
        "An employee message may contain one or several distinct operational issues. "
        "Break it into separate self-contained issues. Return a JSON array of strings, "
        "one per distinct issue, each rewritten as a clear standalone message. "
        "If there is only one issue, return an array with that single message. "
        "Return JSON array only.\n\nMessage: "
    )
    try:
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": split_prompt + message_text}]
        )
        raw = response.content[0].text
        cleaned = re.sub(r"```json|```", "", raw).strip()
        issues = json.loads(cleaned)
        return issues if isinstance(issues, list) and issues else [message_text]
    except Exception:
        return [message_text]   # if splitting fails, treat as one message

def triage_with_splitting(message_text):
    """Full system: split a message into issues, then triage each into its own ticket."""
    issues = split_into_issues(message_text)
    return [triage_validated(issue) for issue in issues]

print("Multi-issue splitting ready.")

Multi-issue splitting ready.


In [59]:
def show_ticket(data):
    print("  ┌────────────────────────────────────────────")
    print(f"  │ Summary    : {data['brief_summary']}")
    print(f"  │ Category   : {data['issue_category']}")
    print(f"  │ Urgency    : {data['urgency']}")
    print(f"  │ Follow-up  : {data['followup_required']}")
    print(f"  │ Sensitivity: {data['data_sensitivity_risk']}")
    print(f"  │ Confidence : {data['confidence']}")
    print(f"  │ ACTION     : {data['routing_action']}")
    print("  └────────────────────────────────────────────")

print("=" * 50)
print("  ESG OPERATIONAL TRIAGE SYSTEM")
print("=" * 50)

total_tickets = 0
auto_routed = 0
escalated = 0

for key, msg in messages.items():
    print(f"\nINCOMING MESSAGE [{key}]:")
    print(f'  "{msg}"')
    tickets = triage_with_splitting(msg)
    if len(tickets) > 1:
        print(f"  >> Split into {len(tickets)} separate tickets:")
    for t in tickets:
        show_ticket(t)
        total_tickets += 1
        if t["routing_action"].startswith("AUTO"):
            auto_routed += 1
        else:
            escalated += 1

print("\n" + "=" * 50)
print("  OPERATIONS SUMMARY")
print("=" * 50)
print(f"  Messages received : {len(messages)}")
print(f"  Tickets generated : {total_tickets}")
print(f"  Auto-routed       : {auto_routed}")
print(f"  Escalated/manual  : {escalated}")

  ESG OPERATIONAL TRIAGE SYSTEM

INCOMING MESSAGE [M1_clear]:
  "There is a water leak in Building C that has been running all morning."
  ┌────────────────────────────────────────────
  │ Summary    : A water leak has been running in Building C since morning and requires immediate attention.
  │ Category   : Water
  │ Urgency    : HIGH
  │ Follow-up  : Y
  │ Sensitivity: NONE
  │ Confidence : 0.92
  │ ACTION     : AUTO-ROUTED to Facilities team
  └────────────────────────────────────────────

INCOMING MESSAGE [M2_emotional]:
  "The recycling bins are contaminated AGAIN and nobody ever checks them. This is ridiculous."
  >> Split into 2 separate tickets:
  ┌────────────────────────────────────────────
  │ Summary    : Recycling bins have been reported as contaminated.
  │ Category   : Waste & Recycling
  │ Urgency    : MEDIUM
  │ Follow-up  : Y
  │ Sensitivity: NONE
  │ Confidence : 0.75
  │ ACTION     : ESCALATED TO HUMAN REVIEW
  └────────────────────────────────────────────
  ┌─────